In [1]:
!pip -q install bitsandbytes sentencepiece openpyxl pypdf sentence-transformers langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb
!pip -q install -U transformers accelerate

import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU YOK! Çalışma zamanı türünü T4 GPU yapın, ortamı sıfırlayıp (bağlantıyı kes ve sil) tekrar deneyin."
print("GPU adı:", torch.cuda.get_device_name(0))

import pandas as pd
import re

from google.colab import files
from pypdf import PdfReader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ------------------------------------------------------------------
# Ayarlar
# ------------------------------------------------------------------
MODEL_NAME     = "Qwen/Qwen3-8B"
MODEL_ETIKET   = "Qwen3_8B_4bit"
MAX_NEW_TOKENS = 512
DO_SAMPLE      = False
REPETITION_PENALTY = 1.1

EMBEDDING_MODEL = "BAAI/bge-m3"
CHUNK_SIZE      = 512
CHUNK_OVERLAP   = 64
TOP_K           = 2

CIKTI_DOSYASI  = f"KVKK_cevaplar_v2_{MODEL_ETIKET}.xlsx"
README_DOSYASI = f"README_experiment_{MODEL_ETIKET}.txt"


print("Questions(in).csv ve KVKK.pdf dosyalarını yükleyin")
uploaded = files.upload()


# ------------------------------------------------------------------
# Model (4-bit sıkıştırılmış, tamamen GPU'da)
# ------------------------------------------------------------------
print("Model yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
    device_map={"": 0}          # modelin tamamını GPU'ya koy; sığmazsa hata verir, CPU'ya kaçmaz
)

print(f"Model GPU'da ✅  (GPU bellek: {torch.cuda.memory_allocated()/1e9:.1f} GB)")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=DO_SAMPLE,
    repetition_penalty=REPETITION_PENALTY,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)


SYSTEM = """
Sen uzman bir hukuk asistanısın.

Türkçe cevap ver.

Kısa ve net cevap ver.

Emin değilsen bunu belirt.
"""


def ask(question, context=None):

    if context:
        user_content = f"""
KAYNAK METİN:

{context}

SORU:

{question}
"""
    else:
        user_content = question

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_content}
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False          # Qwen3: düşünme modunu kapat
    )

    answer = pipe(prompt)[0]["generated_text"]
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S)
    answer = answer.replace("\n", " ").replace("\r", " ")
    answer = " ".join(answer.split())
    return answer.strip()


# Hızlı test: kısa bir cevap görmelisiniz, <think> görmemelisiniz
import time
t = time.time()
print("Test:", ask("Türkiye'nin başkenti neresidir?"), f"({time.time() - t:.1f} sn)")


# ------------------------------------------------------------------
# Vektör veritabanı
# ------------------------------------------------------------------
print("KVKK PDF işleniyor...")

reader = PdfReader("KVKK.pdf")
docs = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        docs.append(Document(page_content=text, metadata={"page": i + 1}))
print("Sayfa sayısı:", len(docs))

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)
print("Chunk sayısı:", len(chunks))

print("Embedding oluşturuluyor...")
embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

try:
    Chroma(collection_name="kvkk", embedding_function=embedding).delete_collection()
except Exception:
    pass

db = Chroma.from_documents(documents=chunks, embedding=embedding, collection_name="kvkk")
retriever = db.as_retriever(search_kwargs={"k": TOP_K})


# ------------------------------------------------------------------
# Sorular
# ------------------------------------------------------------------
df = pd.read_csv("Questions(in).csv", sep=";")
question_col = df.columns[0]

questions = df[question_col].dropna().astype(str).str.strip()
questions = questions[questions.str.len() > 5]
print("Toplam soru:", len(questions))


# ------------------------------------------------------------------
# Cevap üretimi
# ------------------------------------------------------------------
results = []
baslangic = time.time()

for idx, question in enumerate(questions, start=1):

    baseline = ask(question)

    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in retrieved_docs)
    rag = ask(question, context=context)

    pages = ",".join(str(d.metadata.get("page")) for d in retrieved_docs)
    evidence = "\n\n====================\n\n".join(
        f"[Sayfa {d.metadata.get('page')}]\n{d.page_content[:1500]}"
        for d in retrieved_docs
    )

    results.append({
        "question": question,
        "baseline": baseline,
        "rag": rag,
        "retrieved_pages": pages,
        "retrieved_context": evidence
    })

    print(f"Soru {idx}/{len(questions)} tamam ({(time.time() - baslangic)/60:.1f} dk)")


result_df = pd.DataFrame(results)

if "ID" in df.columns:
    result_df.insert(0, "id", df.loc[questions.index, "ID"].values)
else:
    print("UYARI: CSV'de ID sütunu yok; 'id' sütununu elle eklemeniz gerekecek.")

display(result_df)
result_df.to_excel(CIKTI_DOSYASI, index=False)


# ------------------------------------------------------------------
# Deney bilgisi
# ------------------------------------------------------------------
readme = f"""
MODEL
-----
{MODEL_NAME} (4-bit NF4 quantization, bitsandbytes, thinking mode kapalı)

INFERENCE
---------
Transformers Pipeline

PROMPT
------
{SYSTEM.strip()}

GENERATION
----------
max_new_tokens = {MAX_NEW_TOKENS}
do_sample = {DO_SAMPLE}
repetition_penalty = {REPETITION_PENALTY}

RAG
---
Embedding Model : {EMBEDDING_MODEL}
Vector Database : ChromaDB
Chunk Size      : {CHUNK_SIZE}
Chunk Overlap   : {CHUNK_OVERLAP}
Top-K           : {TOP_K}

KNOWLEDGE SOURCE
----------------
KVKK.pdf

QUESTIONS
---------
Questions(in).csv ({len(result_df)} soru, soru seti v2)

EVALUATION
----------
Aynı soru seti hem baseline hem RAG için kullanılmıştır.
Aynı prompt ve aynı model kullanılmıştır.
Tek fark retrieval context'tir.
"""

with open(README_DOSYASI, "w", encoding="utf-8") as f:
    f.write(readme)

print("\n✅ Tamamlandı")

files.download(CIKTI_DOSYASI)
files.download(README_DOSYASI)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
PyTorch: 2.14.1+cu130 | GPU: True
GPU adı: Tesla T4
Questions(in).csv ve KVKK.pdf dosyalarını yükleyin


Saving Questions(in).csv to Questions(in).csv
Saving KVKK.pdf to KVKK.pdf
Model yükleniyor...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (30 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>